In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

In [ ]:
feature_cols = [c for c in df_features.columns if c not in META_COLS]
X        = df_features[feature_cols].values.astype(float)
subjects = df_features["subject_id"].values
datasets = df_features["dataset"].values

le = LabelEncoder()
y  = le.fit_transform(df_features["label"].values)

print("Mapping label:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} -> {cls}")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

param_grids = {
    "MLP": {
        "clf__hidden_layer_sizes": [(64, 32), (128, 64), (64, 32, 16)],
        "clf__alpha"             : [0.0001, 0.001, 0.01],
        "clf__learning_rate_init": [0.001, 0.01],
    },
    "AdaBoost": {
        "clf__n_estimators"         : [100, 200, 300],
        "clf__estimator__max_depth" : [1, 2, 3],
        "clf__learning_rate"        : [0.5, 1.0, 1.5],
    },
    "KNN": {
        "clf__n_neighbors": [3, 5, 7, 11],
        "clf__weights"    : ["uniform", "distance"],
        "clf__metric"     : ["euclidean", "manhattan"],
    },
    "SVM": {
        "clf__C"    : [0.1, 1, 10, 100],
        "clf__gamma": ["scale", "auto", 0.01, 0.001],
    },
}

def build_models():
    return {
        "MLP": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("clf",     MLPClassifier(max_iter=300, early_stopping=True, random_state=42))
        ]),
        "AdaBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("clf",     AdaBoostClassifier(estimator=DecisionTreeClassifier(), random_state=42))
        ]),
        "KNN": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("clf",     KNeighborsClassifier())
        ]),
        "SVM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("clf",     SVC(kernel="rbf", probability=True, random_state=42))
        ]),
    }

model_names = ["MLP", "AdaBoost", "KNN", "SVM"]

In [ ]:
print("=" * 60)
print("CEK PER FOLD - STRATIFIED GROUP 5-FOLD")
print("=" * 60)

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=subjects)):
    test_subj  = set(subjects[test_idx])
    train_subj = set(subjects[train_idx])

    overlap = test_subj & train_subj
    assert len(overlap) == 0, f"LEAKAGE! Subjek overlap di fold {fold+1}: {overlap}"

    idx_overlap = set(test_idx) & set(train_idx)
    assert len(idx_overlap) == 0, f"LEAKAGE! Index overlap di fold {fold+1}"

    assert len(train_idx) + len(test_idx) == len(X), f"Data hilang di fold {fold+1}"

    imputer = SimpleImputer(strategy="median")
    scaler  = StandardScaler()
    X_train_sc = scaler.fit_transform(imputer.fit_transform(X[train_idx]))
    X_test_sc  = scaler.transform(imputer.transform(X[test_idx]))

    test_labels   = pd.Series(y[test_idx]).value_counts().to_dict()
    train_labels  = pd.Series(y[train_idx]).value_counts().to_dict()
    test_dataset  = np.unique(datasets[test_idx])
    test_ds_count = pd.Series(datasets[test_idx]).value_counts().to_dict()

    print(f"\nFold {fold+1}")
    print(f"  Test  subjects  : {len(test_subj)} subjek dari {test_dataset}")
    print(f"  Test  per dataset: {test_ds_count}")
    print(f"  Test  labels    : {test_labels}")
    print(f"  Train labels    : {train_labels}")
    print(f"  Subject overlap : {overlap if overlap else 'TIDAK ADA'}")
    print(f"  3 kelas di test : {'✅' if len(test_labels) == 3 else 'TIDAK LENGKAP'}")

Non- Tuning

In [ ]:
models_baseline = {
    "MLP"     : MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True, random_state=42),
    "AdaBoost": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2), n_estimators=150, random_state=42),
    "KNN"     : KNeighborsClassifier(n_neighbors=7, weights="distance"),
    "SVM"     : SVC(kernel="rbf", probability=True, random_state=42),
}

results                 = {name: [] for name in models_baseline}
all_y_true_baseline     = {name: [] for name in models_baseline}
all_y_pred_baseline     = {name: [] for name in models_baseline}
all_y_prob_baseline     = {name: [] for name in models_baseline}
results_per_ds_baseline = {name: {ds: {"y_true": [], "y_pred": []} for ds in np.unique(datasets)} for name in models_baseline}

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=subjects)):
    test_subs     = np.unique(subjects[test_idx])
    test_datasets = datasets[test_idx]
    print(f"\n── Fold {fold+1} | Test subjects: {len(test_subs)} subjek ──")

    X_train_raw, y_train = X[train_idx], y[train_idx]
    X_test_raw,  y_test  = X[test_idx],  y[test_idx]

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train_imp)
    X_test_sc  = scaler.transform(X_test_imp)

    for name, clf in models_baseline.items():
        clf.fit(X_train_sc, y_train)
        y_pred     = clf.predict(X_test_sc)
        y_prob_raw = clf.predict_proba(X_test_sc)

        all_y_true_baseline[name].extend(y_test.tolist())
        all_y_pred_baseline[name].extend(y_pred.tolist())
        all_y_prob_baseline[name].extend(y_prob_raw.tolist())

        for ds_name in np.unique(test_datasets):
            mask = test_datasets == ds_name
            results_per_ds_baseline[name][ds_name]["y_true"].extend(y_test[mask].tolist())
            results_per_ds_baseline[name][ds_name]["y_pred"].extend(y_pred[mask].tolist())

        results[name].append({
            "fold"     : fold + 1,
            "accuracy" : accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
            "recall"   : recall_score(y_test, y_pred, average="macro", zero_division=0),
            "f1"       : f1_score(y_test, y_pred, average="macro", zero_division=0),
        })

print(f"\n{'Model':<12} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("-" * 55)
for name, folds in results.items():
    acc  = np.mean([f["accuracy"]  for f in folds])
    prec = np.mean([f["precision"] for f in folds])
    rec  = np.mean([f["recall"]    for f in folds])
    f1   = np.mean([f["f1"]        for f in folds])
    print(f"{name:<12} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

print("\n" + "=" * 70)
print("BREAKDOWN PERFORMA PER DATASET (Non-Tuning)")
print("=" * 70)
for name in models_baseline:
    print(f"\n{'─'*70}\nMODEL: {name}\n{'─'*70}")
    for ds_name in np.unique(datasets):
        yt = results_per_ds_baseline[name][ds_name]["y_true"]
        yp = results_per_ds_baseline[name][ds_name]["y_pred"]
        if len(yt) == 0:
            continue
        print(f"\n  Dataset: {ds_name} (n={len(yt)})")
        print(classification_report(yt, yp, labels=np.unique(y),
              target_names=le.classes_, zero_division=0))

Tuning

In [ ]:
def run_tuning_no_selectkbest():
    models = build_models()
    results_local            = {name: [] for name in models}
    all_y_true_local          = {name: [] for name in models}
    all_y_pred_local          = {name: [] for name in models}
    all_y_prob_local          = {name: [] for name in models}
    results_per_ds_local      = {name: {ds: {"y_true": [], "y_pred": []} for ds in np.unique(datasets)} for name in models}

    for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=subjects)):
        test_subs     = np.unique(subjects[test_idx])
        test_datasets = datasets[test_idx]
        print(f"\n── Tuning (no SelectKBest) | Fold {fold+1} | Test subjects: {len(test_subs)} subjek ──")

        X_train_raw, y_train_fold = X[train_idx], y[train_idx]
        X_test_raw,  y_test_fold  = X[test_idx],  y[test_idx]
        subj_train_fold           = subjects[train_idx]

        inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

        for name, pipeline in models.items():
            gs = GridSearchCV(
                pipeline, param_grids[name],
                cv=inner_cv, scoring="f1_macro",
                n_jobs=-1, verbose=0
            )
            gs.fit(X_train_raw, y_train_fold, groups=subj_train_fold)

            y_pred     = gs.best_estimator_.predict(X_test_raw)
            y_prob_raw = gs.best_estimator_.predict_proba(X_test_raw)

            all_y_true_local[name].extend(y_test_fold.tolist())
            all_y_pred_local[name].extend(y_pred.tolist())
            all_y_prob_local[name].extend(y_prob_raw.tolist())

            for ds_name in np.unique(test_datasets):
                mask = test_datasets == ds_name
                results_per_ds_local[name][ds_name]["y_true"].extend(y_test_fold[mask].tolist())
                results_per_ds_local[name][ds_name]["y_pred"].extend(y_pred[mask].tolist())

            results_local[name].append({
                "fold"       : fold + 1,
                "accuracy"   : accuracy_score(y_test_fold, y_pred),
                "precision"  : precision_score(y_test_fold, y_pred, average="macro", zero_division=0),
                "recall"     : recall_score(y_test_fold, y_pred, average="macro", zero_division=0),
                "f1"         : f1_score(y_test_fold, y_pred, average="macro", zero_division=0),
                "best_params": gs.best_params_,
            })
            print(f"  {name:<10} f1_macro={results_local[name][-1]['f1']:.4f}  {gs.best_params_}")

    print(f"\n{'Model':<12} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
    print("─" * 55)
    for name, folds in results_local.items():
        acc  = np.mean([f["accuracy"]  for f in folds])
        prec = np.mean([f["precision"] for f in folds])
        rec  = np.mean([f["recall"]    for f in folds])
        f1   = np.mean([f["f1"]        for f in folds])
        print(f"{name:<12} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

    print("\n" + "=" * 70)
    print("BREAKDOWN PERFORMA PER DATASET (Tuning tanpa SelectKBest)")
    print("=" * 70)
    for name in models:
        print(f"\n{'─'*70}\nMODEL: {name}\n{'─'*70}")
        for ds_name in np.unique(datasets):
            yt = results_per_ds_local[name][ds_name]["y_true"]
            yp = results_per_ds_local[name][ds_name]["y_pred"]
            if len(yt) == 0:
                continue
            print(f"\n  Dataset: {ds_name} (n={len(yt)})")
            print(classification_report(yt, yp, labels=np.unique(y),
                  target_names=le.classes_, zero_division=0))

    return results_local, all_y_true_local, all_y_pred_local, all_y_prob_local, results_per_ds_local

Tuning With Select KBest

In [ ]:
def run_tuning(K_FEATURES):
    models = build_models()
    results_local            = {name: [] for name in models}
    all_y_true_local          = {name: [] for name in models}
    all_y_pred_local          = {name: [] for name in models}
    all_y_prob_local          = {name: [] for name in models}
    feature_selection_count   = np.zeros(len(feature_cols))
    results_per_ds_local      = {name: {ds: {"y_true": [], "y_pred": []} for ds in np.unique(datasets)} for name in models}

    for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=subjects)):
        test_subs     = np.unique(subjects[test_idx])
        test_datasets = datasets[test_idx]
        print(f"\n── K={K_FEATURES} | Fold {fold+1} | Test subjects: {len(test_subs)} subjek ──")

        X_train_raw, y_train_fold = X[train_idx], y[train_idx]
        X_test_raw,  y_test_fold  = X[test_idx],  y[test_idx]
        subj_train_fold           = subjects[train_idx]

        imputer = SimpleImputer(strategy="median")
        scaler  = StandardScaler()
        X_train_sc = scaler.fit_transform(imputer.fit_transform(X_train_raw))
        X_test_sc  = scaler.transform(imputer.transform(X_test_raw))

        fold_selector = SelectKBest(f_classif, k=K_FEATURES)
        fold_selector.fit(X_train_sc, y_train_fold)
        feature_selection_count += fold_selector.get_support().astype(int)
        X_train_sel = fold_selector.transform(X_train_sc)
        X_test_sel  = fold_selector.transform(X_test_sc)

        inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

        for name, pipeline in models.items():
            gs = GridSearchCV(
                pipeline, param_grids[name],
                cv=inner_cv, scoring="f1_macro",
                n_jobs=-1, verbose=0
            )
            gs.fit(X_train_sel, y_train_fold, groups=subj_train_fold)

            y_pred     = gs.best_estimator_.predict(X_test_sel)
            y_prob_raw = gs.best_estimator_.predict_proba(X_test_sel)

            all_y_true_local[name].extend(y_test_fold.tolist())
            all_y_pred_local[name].extend(y_pred.tolist())
            all_y_prob_local[name].extend(y_prob_raw.tolist())

            for ds_name in np.unique(test_datasets):
                mask = test_datasets == ds_name
                results_per_ds_local[name][ds_name]["y_true"].extend(y_test_fold[mask].tolist())
                results_per_ds_local[name][ds_name]["y_pred"].extend(y_pred[mask].tolist())

            results_local[name].append({
                "fold"       : fold + 1,
                "accuracy"   : accuracy_score(y_test_fold, y_pred),
                "precision"  : precision_score(y_test_fold, y_pred, average="macro", zero_division=0),
                "recall"     : recall_score(y_test_fold, y_pred, average="macro", zero_division=0),
                "f1"         : f1_score(y_test_fold, y_pred, average="macro", zero_division=0),
                "best_params": gs.best_params_,
            })
            print(f"  {name:<10} f1_macro={results_local[name][-1]['f1']:.4f}  {gs.best_params_}")

    print(f"\n{'Model':<12} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
    print("─" * 55)
    for name, folds in results_local.items():
        acc  = np.mean([f["accuracy"]  for f in folds])
        prec = np.mean([f["precision"] for f in folds])
        rec  = np.mean([f["recall"]    for f in folds])
        f1   = np.mean([f["f1"]        for f in folds])
        print(f"{name:<12} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f}")

    print("\n" + "=" * 70)
    print(f"BREAKDOWN PERFORMA PER DATASET (Tuning+SelectKBest K={K_FEATURES})")
    print("=" * 70)
    for name in models:
        print(f"\n{'─'*70}\nMODEL: {name}\n{'─'*70}")
        for ds_name in np.unique(datasets):
            yt = results_per_ds_local[name][ds_name]["y_true"]
            yp = results_per_ds_local[name][ds_name]["y_pred"]
            if len(yt) == 0:
                continue
            print(f"\n  Dataset: {ds_name} (n={len(yt)})")
            print(classification_report(yt, yp, labels=np.unique(y),
                  target_names=le.classes_, zero_division=0))

    return results_local, all_y_true_local, all_y_pred_local, all_y_prob_local, feature_selection_count, results_per_ds_local

k=15

In [ ]:
results_k15, all_y_true_k15, all_y_pred_k15, all_y_prob_k15, feature_selection_count_k15, results_per_ds_k15 = run_tuning(K_FEATURES=15)

k=20

In [ ]:
results_k20, all_y_true_k20, all_y_pred_k20, all_y_prob_k20, feature_selection_count_k20, results_per_ds_k20 = run_tuning(K_FEATURES=20)

k=25

In [ ]:
results_k25, all_y_true_k25, all_y_pred_k25, all_y_prob_k25, feature_selection_count_k25, results_per_ds_k25 = run_tuning(K_FEATURES=25)

comparison of schemes

In [ ]:
all_schemes = {
    "Non-tuning"        : (results,           all_y_true_baseline,  all_y_pred_baseline,  all_y_prob_baseline),
    "Tuning only"       : (results_tune_only,  all_y_true_tune_only, all_y_pred_tune_only, all_y_prob_tune_only),
    "Tuning+SelectK=15" : (results_k15,        all_y_true_k15,       all_y_pred_k15,       all_y_prob_k15),
    "Tuning+SelectK=20" : (results_k20,        all_y_true_k20,       all_y_pred_k20,       all_y_prob_k20),
    "Tuning+SelectK=25" : (results_k25,        all_y_true_k25,       all_y_pred_k25,       all_y_prob_k25),
}

print(f"\n{'Skema':<20} " + "".join(f"{name:>12}" for name in model_names) + f"{'Rata-rata':>14}")
print("-" * (20 + 12 * len(model_names) + 14))

scheme_avg_f1 = {}
for scheme_name, (res, yt, yp, ypr) in all_schemes.items():
    f1_per_model = [np.mean([f["f1"] for f in res[name]]) for name in model_names]
    avg_f1 = np.mean(f1_per_model)
    scheme_avg_f1[scheme_name] = avg_f1
    print(f"{scheme_name:<20} " + "".join(f"{f1:>12.4f}" for f1 in f1_per_model) + f"{avg_f1:>14.4f}")

best_scheme_name = max(scheme_avg_f1, key=scheme_avg_f1.get)

print("\n" + "=" * 60)
print(f"SKEMA TERBAIK: {best_scheme_name} (rata-rata F1-macro = {scheme_avg_f1[best_scheme_name]:.4f})")
print("=" * 60)

results_final, all_y_true_final, all_y_pred_final, all_y_prob_final = all_schemes[best_scheme_name]

In [ ]:
all_schemes = {
    "Non-tuning"        : (results,           all_y_true_baseline,  all_y_pred_baseline,  all_y_prob_baseline),
    "Tuning only"       : (results_tune_only,  all_y_true_tune_only, all_y_pred_tune_only, all_y_prob_tune_only),
    "Tuning+SelectK=15" : (results_k15,        all_y_true_k15,       all_y_pred_k15,       all_y_prob_k15),
    "Tuning+SelectK=20" : (results_k20,        all_y_true_k20,       all_y_pred_k20,       all_y_prob_k20),
    "Tuning+SelectK=25" : (results_k25,        all_y_true_k25,       all_y_pred_k25,       all_y_prob_k25),
}

print(f"\n{'Skema':<20} " + "".join(f"{name:>12}" for name in model_names) + f"{'Rata-rata':>14}")
print("-" * (20 + 12 * len(model_names) + 14))

scheme_avg_f1 = {}
for scheme_name, (res, yt, yp, ypr) in all_schemes.items():
    f1_per_model = [np.mean([f["f1"] for f in res[name]]) for name in model_names]
    avg_f1 = np.mean(f1_per_model)
    scheme_avg_f1[scheme_name] = avg_f1
    print(f"{scheme_name:<20} " + "".join(f"{f1:>12.4f}" for f1 in f1_per_model) + f"{avg_f1:>14.4f}")

best_scheme_name = max(scheme_avg_f1, key=scheme_avg_f1.get)

print("\n" + "=" * 60)
print(f"SKEMA TERBAIK: {best_scheme_name} (rata-rata F1-macro = {scheme_avg_f1[best_scheme_name]:.4f})")
print("=" * 60)

results_final, all_y_true_final, all_y_pred_final, all_y_prob_final = all_schemes[best_scheme_name]

In [ ]:
best_scheme_name = "Non-tuning"

results_final, all_y_true_final, all_y_pred_final, all_y_prob_final = all_schemes[best_scheme_name]

EVALUATION & VISUALISATION

In [ ]:
#CONFUSION MATRIX

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, name in enumerate(model_names):
    cm = confusion_matrix(all_y_true_final[name], all_y_pred_final[name], labels=np.unique(y))
    disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
    disp.plot(ax=axes[i], colorbar=False, cmap="Blues", xticks_rotation=30)
    axes[i].set_title(name, fontweight="bold")

plt.suptitle(f"Confusion Matrix", fontsize=14, fontweight="bold")
...

In [ ]:
#AUC-ROC

n_classes = len(le.classes_)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, name in enumerate(model_names):
    ax = axes[i]
    y_true_arr = np.array(all_y_true_final[name])
    y_prob_arr = np.array(all_y_prob_final[name])
    y_true_bin = label_binarize(y_true_arr, classes=np.arange(n_classes))

    for c in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, c], y_prob_arr[:, c])
        auc_score   = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{le.classes_[c]} (AUC={auc_score:.3f})")

    ax.plot([0, 1], [0, 1], "k--", label="Random (AUC=0.500)")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(name, fontweight="bold")
    ax.legend(fontsize=8)

plt.suptitle(f"ROC Curve", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(data_path, "roc_curve_final.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

n_classes = len(le.classes_)

for name in model_names:
    y_true_arr = np.array(all_y_true_final[name])
    y_prob_arr = np.array(all_y_prob_final[name])
    y_true_bin = label_binarize(y_true_arr, classes=np.arange(n_classes))

    # hitung ROC tiap kelas dulu
    fpr_dict, tpr_dict = {}, {}
    for c in range(n_classes):
        fpr_dict[c], tpr_dict[c], _ = roc_curve(y_true_bin[:, c], y_prob_arr[:, c])

    # gabungin semua fpr jadi 1 grid biar bisa dirata-rata (macro-average)
    all_fpr = np.unique(np.concatenate([fpr_dict[c] for c in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr_dict[c], tpr_dict[c])
    mean_tpr /= n_classes

    macro_auc = auc(all_fpr, mean_tpr)
    ax.plot(all_fpr, mean_tpr, label=f"{name} (macro-AUC={macro_auc:.3f})", linewidth=2)

ax.plot([0, 1], [0, 1], "k--", label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curve", fontweight="bold")
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(data_path, "roc_curve_semua_model.png"), dpi=150, bbox_inches="tight")
plt.show()